# 🚦 IntelliDrive AI: Model Training & Evaluation

This notebook covers the training process for the CNN Traffic Sign Classifier and the YOLO Object Detection integration.

In [ ]:
import os
from ultralytics import YOLO
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score

# Reinforcement Learning Libraries
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env

# Set paths (relative to notebook in experiments/ folder)
DATASET_PATH = '../dataset2/traffic_Data/DATA'
TEST_PATH = '../dataset2/traffic_Data/TEST'
LABELS_PATH = '../dataset2/labels.csv'
print(f"Using dataset at: {DATASET_PATH}")

## 1. Data Analysis (Statistics)
Calculating mean, median, and mode for image distributions.

In [ ]:
def get_stats(data_path, test_path):
    if not os.path.exists(data_path):
        print(f"Error: Path {data_path} does not exist.")
        return {}, []
    
    classes = os.listdir(data_path)
    print(f"Found {len(classes)} classes in DATA folder.")
    
    # Gather all image paths and labels for RL environment mapping
    all_images = []
    class_to_idx = {cls: idx for idx, cls in enumerate(sorted(classes))}
    idx_to_class = {idx: cls for cls, idx in class_to_idx.items()}
    
    total_train = 0
    for cls in classes:
        cls_dir = os.path.join(data_path, cls)
        imgs = [f for f in os.listdir(cls_dir) if f.endswith('.png') or f.endswith('.jpg')]
        total_train += len(imgs)
        for img in imgs:
            all_images.append((os.path.join(cls_dir, img), class_to_idx[cls]))
            
    total_test = len(os.listdir(test_path)) if os.path.exists(test_path) else 0
    
    print(f"Total training images: {total_train}")
    print(f"Total test images: {total_test}")
    
    label_map = {}
    if os.path.exists(LABELS_PATH):
        labels_df = pd.read_csv(LABELS_PATH)
        label_map = dict(zip(labels_df['ClassId'].astype(str), labels_df['Name']))
        print(f"Loaded {len(label_map)} class names from labels.csv")
        
    return label_map, all_images, class_to_idx, idx_to_class

label_map, train_images_list, class_to_idx, idx_to_class = get_stats(DATASET_PATH, TEST_PATH)

## 2. YOLOv8 Training
Training YOLO with the custom dataset.

## 2. YOLOv8 Classification Training
YOLOv8 can also be used as a high-speed Image Classifier. We will train it directly on the `dataset2/traffic_Data/DATA` directories.

In [ ]:
import os
from ultralytics import YOLO

# Build paths locally to notebook
DATASET_PATH_YOLO = os.path.abspath(DATASET_PATH)
print(f"YOLO Training on: {DATASET_PATH_YOLO}")

# Load a YOLOv8 classification model
yolo_model = YOLO('yolov8n-cls.pt')

# Train the model for image classification
# (Using 5 epochs for speed of demonstration)
results = yolo_model.train(
    data=DATASET_PATH_YOLO, 
    epochs=5, 
    imgsz=64, 
    batch=64
)
print("YOLOv8 Classification Training Complete!")

In [ ]:
# Visualize YOLO metrics
from IPython.display import Image, display

runs_dir = 'runs/classify'
latest_train_dir = None
if os.path.exists(runs_dir):
    train_dirs = [os.path.join(runs_dir, d) for d in os.listdir(runs_dir) if d.startswith('train')]
    if train_dirs:
        latest_train_dir = max(train_dirs, key=os.path.getmtime)
        print(f"Using analytics from: {latest_train_dir}")

if latest_train_dir and os.path.exists(os.path.join(latest_train_dir, 'results.png')):
    print("Training Metrics:")
    display(Image(filename=os.path.join(latest_train_dir, 'results.png'), width=800))
if latest_train_dir and os.path.exists(os.path.join(latest_train_dir, 'confusion_matrix.png')):
    print("Confusion Matrix:")
    display(Image(filename=os.path.join(latest_train_dir, 'confusion_matrix.png'), width=800))


## 3. Reinforcement Learning (PPO) for Traffic Sign Classification
Formulating the image classification task as a Reinforcement Learning environment where the agent attempts to guess the class.

In [ ]:
import cv2
import numpy as np
import gymnasium as gym
from gymnasium import spaces

class TrafficSignEnv(gym.Env):
    """Custom Environment that follows gym interface for Image Classification"""
    metadata = {'render.modes': ['console']}

    def __init__(self, images_list, num_classes):
        super(TrafficSignEnv, self).__init__()
        self.images_list = images_list
        self.num_classes = num_classes
        self.current_step = 0
        self.max_steps = 1000 # Episodes span 1000 uniform samples
        
        # Action space: Predict one of the classes
        self.action_space = spaces.Discrete(self.num_classes)
        
        # Observation space: 64x64 RGB images as uint8
        self.observation_space = spaces.Box(low=0, high=255, shape=(64, 64, 3), dtype=np.uint8)
        self.current_image = None
        self.current_label = None
        
    def _load_random_image(self):
        img_path, label = random.choice(self.images_list)
        
        # Load and resize using OpenCV
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (64, 64))
        
        self.current_image = img
        self.current_label = label
        return self.current_image

    def step(self, action):
        # Check if the predicted class matches the actual label
        if action == self.current_label:
            reward = 1.0
        else:
            reward = -1.0
            
        self.current_step += 1
        done = self.current_step >= self.max_steps
        truncated = False
        
        # Get next state
        next_state = self._load_random_image()
        
        return next_state, reward, done, truncated, {}

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0
        # Load a random image to start
        obs = self._load_random_image()
        return obs, {}

num_classes = len(class_to_idx)
env = TrafficSignEnv(train_images_list, num_classes)

# Verify the custom environment
check_env(env, warn=True)
print("TrafficSignEnv successfully created and verified.")

In [ ]:
from stable_baselines3 import PPO

# Initialize the RL Agent with a CnnPolicy suited for images
print("Initializing PPO agent...")
model = PPO("CnnPolicy", env, verbose=1, learning_rate=0.0003, batch_size=64)

# Train the agent (Using 10,000 timesteps for demonstration, increase for better accuracy)
print("Training PPO agent...")
model.learn(total_timesteps=10000)

print("Agent trained successfully!")
# Save the trained agent
model.save("ppo_traffic_sign_classifier")

In [ ]:
import os
from IPython.display import Image, display

# Find the most recent training run directory
runs_dir = 'runs/detect'
latest_train_dir = None

if os.path.exists(runs_dir):
    train_dirs = [os.path.join(runs_dir, d) for d in os.listdir(runs_dir) if d.startswith('train')]
    if train_dirs:
        latest_train_dir = max(train_dirs, key=os.path.getmtime)
        print(f"Using analytics from: {latest_train_dir}")
else:
    print("Training run directory not found. Please run the training cell first.")

## 6. Test RL Agent on Dataset2 TEST Images & Metrics
We'll use our trained PPO agent to predict the class for images in the TEST folder and measure its F1-Score/Accuracy.
*Note: RL performs supervised classification by picking actions given environmental states (images).*

In [ ]:
all_preds = []
all_labels = []

test_images = [f for f in os.listdir(TEST_PATH) if f.endswith('.png') or f.endswith('.jpg')]
print(f"Evaluating on {len(test_images)} test images...")

test_display_images = []
for i, img_name in enumerate(test_images):
    img_path = os.path.join(TEST_PATH, img_name)
    
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img, (64, 64))
    
    # Predict action using RL agent
    action, _state = model.predict(img_resized, deterministic=True)
    
    # Parse ground truth from filename
    try:
        class_str = str(int(img_name.split('_')[0]))
        true_label = class_to_idx.get(class_str, 0)
    except:
        true_label = 0
        
    all_preds.append(int(action))
    all_labels.append(true_label)
    
    # Save a few samples for display
    if i < 5:
        test_display_images.append((img_resized, int(action), true_label))

# Calculate metrics
acc = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
print(f"\nTest Accuracy: {acc*100:.2f}%")
print(f"Test F1 Score (weighted): {f1:.4f}")

# Visual predictions
plt.figure(figsize=(15, 5))
for i, (img, pred_idx, true_idx) in enumerate(test_display_images):
    pred_class_str = idx_to_class.get(pred_idx, str(pred_idx))
    true_class_str = idx_to_class.get(true_idx, str(true_idx))
    
    pred_name = label_map.get(pred_class_str, pred_class_str)
    true_name = label_map.get(true_class_str, true_class_str)
    
    plt.subplot(1, 5, i+1)
    plt.imshow(img)
    plt.title(f"T: {true_name}\nP: {pred_name}", fontsize=10)
    plt.axis('off')
    
plt.tight_layout()
plt.show()